# OPD / OPD+ on a single A100

On-policy distillation of **Qwen3-4B-Instruct-2507 -> Qwen3-0.6B-Base** on GSM8K, comparing the
stop-gradient advantage `-f(u)` used by the Thinking Machines recipe against the
gradient-faithful advantage `w_f(u) = -f(u) + u f'(u)` from OPD+ (arXiv:2606.01039).

Runtime -> Change runtime type -> **A100 GPU**. Total budget is about two hours of GPU time.

In [ ]:
!nvidia-smi
import torch, platform
print(platform.python_version(), torch.__version__, torch.cuda.get_device_name(0))

## 1. Get the code

Set `REPO_URL` if you pushed the project to git. Otherwise upload the folder to
`/content/opd` (Files pane, drag and drop) and just run the `%cd`.

In [ ]:
import os, subprocess

REPO_URL = ""  # e.g. "https://github.com/<you>/opd-distillation.git"
PROJECT = "/content/opd"

if REPO_URL and not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)

%cd $PROJECT
!ls

In [ ]:
!pip install -q -U "transformers>=4.51.0" "datasets>=2.19.0" accelerate pytest
# Colab ships a recent torch already; do not reinstall it, that costs 5 minutes.

## 2. Math check before spending any GPU time

Validates the advantage table against autograd. Runs on CPU in a couple of seconds.

In [ ]:
!python -m pytest tests -q

## 3. Smoke test the whole pipeline (~8 min)

Tiny SFT, 2 on-policy steps, 40 eval questions. Catches OOM, tokenizer mismatch and
alignment bugs before the real run.

In [ ]:
!python run_all.py --tier smoke --results-dir results_smoke

## 4. Full grid

Nine arms in sequence, resumable: rerunning skips anything with a `*_final.json`.
Output is tailed to `train.log` so a browser disconnect does not lose it.

In [ ]:
!nohup python run_all.py --tier all --steps 40 --eval-limit 200 > train.log 2>&1 &
print("launched")

In [ ]:
# Re-run this cell to follow progress.
!tail -n 30 train.log
!ls results/

## 5. Figures and table

In [ ]:
!python make_plots.py --results-dir results --out-dir figures

from IPython.display import Image, display
import glob
for path in sorted(glob.glob("figures/*.png")):
    print(path)
    display(Image(path))

## 6. Save results back to Drive

Colab VMs are ephemeral. Copy `results/` and `figures/` out before the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/opd_run && cp -r results figures train.log /content/drive/MyDrive/opd_run/
!ls /content/drive/MyDrive/opd_run